# 01 — Data Collection

Загружаем городскую инфраструктуру из OpenStreetMap и сохраняем в `data/raw/`.

Что должно быть в ноутбуке:
- выбор города и bounding box / place name
- выбор категорий OSM (cafe, restaurant, pharmacy, bar, ...)
- загрузка через `osmnx.features_from_place`
- базовая очистка (координаты, name, category)
- сохранение в CSV / GeoJSON

In [81]:
import osmnx as ox
import pandas as pd
from pathlib import Path

RAW = Path('../data/raw')
RAW.mkdir(parents=True, exist_ok=True)

In [82]:
# TODO: choose place and categories
place = 'Yerevan, Armenia'
tags = {'amenity': ['cafe', 'restaurant', 'pharmacy', 'bar']}

gdf = ox.features_from_place(place, tags=tags)
gdf.shape
column_report = pd.DataFrame({
    'Column_Name': gdf.columns,
    'Non_Null_Count': gdf.count().values,
    'Fill_Rate_%': (gdf.count().values / len(gdf)) * 100,
    'Dtype': gdf.dtypes.values,
    'Example_Value': [gdf[col].dropna().iloc[0] for col in gdf.columns]
})

column_report = column_report.sort_values('Non_Null_Count', ascending=False)

with pd.option_context('display.max_rows', None):
    display(column_report)

for col in gdf["geometry"]:
    print(col)

,Column_Name,Non_Null_Count,Fill_Rate_%,Dtype,Example_Value
0,geometry,1613,100.000000,geometry,POINT (44.5248669 40.1783161)
5,amenity,1613,100.000000,object,restaurant
10,name,1215,75.325480,object,Օլդ Մարանի
6,check_date,933,57.842529,object,2024-11-01
11,name:en,670,41.537508,object,Old Marani
14,opening_hours,639,39.615623,object,Mo-Su 10:00-00:00
4,addr:street,602,37.321761,object,Ալեք Մանուկյանի փողոց
3,addr:housenumber,572,35.461872,object,7
7,cuisine,475,29.448233,object,georgian;armenian
1,addr:city,462,28.642281,object,Երևան


POINT (44.5248669 40.1783161)
POINT (44.5244171 40.1786208)
POINT (44.5191239 40.188278)
POINT (44.5132882 40.1702504)
POINT (44.517692 40.183031)
POINT (44.5235664 40.1778934)
POINT (44.523888 40.1773437)
POINT (44.5140073 40.1889084)
POINT (44.5152042 40.187883)
POINT (44.5164919 40.1848583)
POINT (44.5179639 40.1888541)
POINT (44.5178141 40.1854041)
POINT (44.5137851 40.1853834)
POINT (44.5208333 40.1857071)
POINT (44.5107072 40.1851173)
POINT (44.5150363 40.1838787)
POINT (44.5242844 40.130388)
POINT (44.524653 40.1307286)
POINT (44.5141671 40.1761559)
POINT (44.5148069 40.181336)
POINT (44.4467752 40.1796973)
POINT (44.5164143 40.1637902)
POINT (44.4634806 40.1458151)
POINT (44.462324 40.1449749)
POINT (44.5124995 40.1633825)
POINT (44.5329208 40.1924127)
POINT (44.5510099 40.198349)
POINT (44.5191253 40.1902325)
POINT (44.4989674 40.180493)
POINT (44.4979911 40.1806243)
POINT (44.5105958 40.1553672)
POINT (44.5151704 40.1846738)
POINT (44.524557 40.1888895)
POINT (44.5178765 40.1

In [83]:
cols = [c for c in ["name", "amenity", "geometry", "cuisine", "check_date", "addr:city"] if c in gdf.columns]
gdf = gdf[cols].copy()
gdf

name     amenity  \
element id                                                          
node    370052673                          Օլդ Մարանի  restaurant   
        370052735                             Ֆլագման        cafe   
        370055074                             Առագաստ  restaurant   
        370064207                       Pizza di Roma  restaurant   
        370148312                              Գուստո  restaurant   
...                                               ...         ...   
way     1416288209                    Sorpreso coffee        cafe   
        1430212881  202 °F Alternative Coffee Brewing        cafe   
        1449380224                                NaN    pharmacy   
        1458860288                                NaN        cafe   
        1458860289                                NaN        cafe   

                                                             geometry  \
element id                                                              
node    370052673                           POINT (44.52487 40.17832)   
        370052735                           POINT (44.52442 40.17862)   
        370055074                           POINT (44.51912 40.18828)   
        370064207                           POINT (44.51329 40.17025)   
        370148312                           POINT (44.51769 40.18303)   
...                                                               ...   
way     1416288209  POLYGON ((44.51665 40.18804, 44.51665 40.18802...   
        1430212881  POLYGON ((44.51916 40.18783, 44.51925 40.1878,...   
        1449380224  POLYGON ((44.46804 40.20814, 44.46805 40.2082,...   
        1458860288  POLYGON ((44.52753 40.18165, 44.52755 40.18165...   
        1458860289  POLYGON ((44.52753 40.1817, 44.52755 40.1817, ...   

                              cuisine  check_date addr:city  
element id                                                   
node    370052673   georgian;armenian  2024-11-01     Երևան  
        370052735                 NaN  2023-05-28       NaN  
        370055074                 NaN  2023-09-03       NaN  
        370064207               pizza  2022-05-19     Երևան  
        370148312             italian         NaN       NaN  
...                               ...         ...       ...  
way     1416288209        coffee_shop  2025-03-16       NaN  
        1430212881        coffee_shop  2023-09-10       NaN  
        1449380224                NaN         NaN       NaN  
        1458860288        coffee_shop         NaN       NaN  
        1458860289        coffee_shop         NaN       NaN  

[1613 rows x 6 columns]